# Financial Data RAG System with FAISS

This notebook demonstrates building a complete Retrieval-Augmented Generation (RAG) system for financial data analysis using FAISS embeddings and Google Gemini LLM.

## Overview
- **Data Sources**: Holdings and Trades CSV files
- **Embedding Model**: SentenceTransformer (all-mpnet-base-v2)
- **Vector Database**: FAISS with normalized embeddings
- **LLM Integration**: Google Gemini for intelligent responses
- **Search Capabilities**: Semantic search across financial records

## System Architecture
1. Document Preprocessing and Chunking
2. Embedding Generation
3. FAISS Index Creation
4. Semantic Search Implementation
5. RAG-powered Chatbot Integration

In [8]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import pickle
from typing import Dict, List, Tuple
import json

# Load the CSV files
holdings_df = pd.read_csv('holdings.csv')
trades_df = pd.read_csv('trades.csv')

print(f"Holdings shape: {holdings_df.shape}")
print(f"Trades shape: {trades_df.shape}")
print("\nHoldings columns:", list(holdings_df.columns))
print("\nTrades columns:", list(trades_df.columns))

<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type swigvarlink has no __module__ attribute
c:\Users\akash\Documents\Projects\freelance\propertyLoop\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Holdings shape: (1022, 25)
Trades shape: (649, 31)

Holdings columns: ['AsOfDate', 'OpenDate', 'CloseDate', 'ShortName', 'PortfolioName', 'StrategyRefShortName', 'Strategy1RefShortName', 'Strategy2RefShortName', 'CustodianName', 'DirectionName', 'SecurityId', 'SecurityTypeName', 'SecName', 'StartQty', 'Qty', 'StartPrice', 'Price', 'StartFXRate', 'FXRate', 'MV_Local', 'MV_Base', 'PL_DTD', 'PL_QTD', 'PL_MTD', 'PL_YTD']

Trades columns: ['id', 'RevisionId', 'AllocationId', 'TradeTypeName', 'SecurityId', 'SecurityType', 'Name', 'Ticker', 'CUSIP', 'ISIN', 'TradeDate', 'SettleDate', 'Quantity', 'Price', 'TradeFXRate', 'Principal', 'Interest', 'TotalCash', 'AllocationQTY', 'AllocationPrincipal', 'AllocationInterest', 'AllocationFees', 'AllocationCash', 'PortfolioName', 'CustodianName', 'StrategyName', 'Strategy1Name', 'Strategy2Name', 'Counterparty', 'AllocationRule', 'IsCustomAllocation']


## Step 1: Data Loading and Initial Setup

Load the financial datasets and examine their structure to understand the data schema.

In [9]:
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    import sentence_transformers
    import faiss
except ImportError:
    print("Installing required packages...")
    install_package("sentence-transformers")
    install_package("faiss-cpu")
    print("Installation complete!")

## Step 2: Package Installation

Install required packages for sentence transformers and FAISS vector database.

In [10]:
class TabularDataProcessor:
    def __init__(self, model_name='all-mpnet-base-v2'):
        """
        Initialize with a sentence transformer model that works well for tabular data
        all-mpnet-base-v2 is excellent for semantic similarity and works well with structured data
        """
        self.model = SentenceTransformer(model_name)
        self.embedding_dimension = self.model.get_sentence_embedding_dimension()
        
    def preprocess_row(self, row: pd.Series, dataset_type: str) -> str:
        """
        Convert a pandas row to a text representation suitable for embedding
        Preserves all important information while creating meaningful text
        """
        # Remove NaN values and convert to string
        clean_row = {k: v for k, v in row.items() if pd.notna(v)}
        
        # Create a structured text representation
        text_parts = [f"Dataset: {dataset_type}"]
        
        # Add key-value pairs in a structured format
        for key, value in clean_row.items():
            if key.lower() in ['id', 'securityid', 'allocationid']:
                text_parts.append(f"ID_{key}: {value}")
            elif key.lower() in ['name', 'ticker', 'securitytype']:
                text_parts.append(f"Security_{key}: {value}")
            elif key.lower() in ['portfolioname', 'custodianname', 'strategyname']:
                text_parts.append(f"Portfolio_{key}: {value}")
            elif 'date' in key.lower():
                text_parts.append(f"Date_{key}: {value}")
            elif key.lower() in ['quantity', 'price', 'principal', 'totalcash']:
                text_parts.append(f"Financial_{key}: {value}")
            else:
                text_parts.append(f"{key}: {value}")
        
        return " | ".join(text_parts)
    
    def create_metadata(self, row: pd.Series, row_index: int, dataset_type: str) -> Dict:
        """Create comprehensive metadata for each row"""
        metadata = {
            'row_index': row_index,
            'dataset_type': dataset_type,
            'original_data': row.to_dict()
        }
        
        # Add key identifiers
        if 'id' in row.index:
            metadata['primary_id'] = str(row['id'])
        if 'SecurityId' in row.index:
            metadata['security_id'] = str(row['SecurityId'])
        if 'PortfolioName' in row.index:
            metadata['portfolio_name'] = str(row['PortfolioName'])
        if 'Name' in row.index:
            metadata['security_name'] = str(row['Name'])
            
        return metadata

# Initialize the processor
processor = TabularDataProcessor()
print(f"Model loaded. Embedding dimension: {processor.embedding_dimension}")

c:\Users\akash\Documents\Projects\freelance\propertyLoop\env\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\akash\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 455.91it/s, 

Model loaded. Embedding dimension: 768


## Step 3: Document Preprocessing and Text Representation

Create a processor class that converts tabular data into structured text suitable for embedding generation. This step is crucial for maintaining semantic relationships in financial data.

In [ ]:
# Process both datasets and create embeddings
print("Processing Holdings data...")
holdings_texts = []
holdings_metadata = []

for idx, row in holdings_df.iterrows():
    text = processor.preprocess_row(row, "holdings")
    metadata = processor.create_metadata(row, idx, "holdings")
    holdings_texts.append(text)
    holdings_metadata.append(metadata)

print("Processing Trades data...")
trades_texts = []
trades_metadata = []

for idx, row in trades_df.iterrows():
    text = processor.preprocess_row(row, "trades")
    metadata = processor.create_metadata(row, idx, "trades")
    trades_texts.append(text)
    trades_metadata.append(metadata)

print(f"Processed {len(holdings_texts)} holdings records and {len(trades_texts)} trades records")

# Example of processed text (first holding)
print("\nExample processed holdings text:")
print(holdings_texts[0][:200] + "..." if len(holdings_texts[0]) > 200 else holdings_texts[0])

# Example of processed text (first trade)
print("\nExample processed trades text:")
print(trades_texts[0][:200] + "..." if len(trades_texts[0]) > 200 else trades_texts[0])

Processing Holdings data...
Processing Trades data...
Processed 1022 holdings records and 649 trades records

Example processed holdings text:
Dataset: holdings | Date_AsOfDate: 01/08/23 | Date_OpenDate: 04/03/20 | ShortName: Garfield | Portfolio_PortfolioName: Garfield | StrategyRefShortName:  Default | Strategy1RefShortName: Asset | Strategy2RefShortName: DefaultS2 | Portfolio_CustodianName: Well Prime | DirectionName: Long | ID_SecurityId: 273098 | SecurityTypeName: Bond | SecName: EJ0445951 | StartQty: 592000.0 | Qty: 592000.0 | StartPrice: 96.0 | Financial_Price: 96.0 | StartFXRate: 1.33 | FXRate: 1.33 | MV_Local: 568320.0 | MV_Ba...

Example processed trades text:
Dataset: trades | ID_id: 3489863 | RevisionId: 2 | ID_AllocationId: 3460886 | TradeTypeName: Buy | ID_SecurityId: 270471 | Security_SecurityType: Equity | Security_Name: Berry Brand 4/11 Equity | Date_TradeDate: 00:00.0 | Date_SettleDate: 00:00.0 | Financial_Quantity: 500000 | Financial_Price: 14.0 | Financial_Principal

## Step 4: Data Processing and Metadata Creation

Process both datasets to create structured text representations and comprehensive metadata for each record.

In [12]:
# Create embeddings for all data
print("Creating embeddings...")
all_texts = holdings_texts + trades_texts
all_metadata = holdings_metadata + trades_metadata

# Generate embeddings in batches to manage memory
batch_size = 32
all_embeddings = []

for i in range(0, len(all_texts), batch_size):
    batch_texts = all_texts[i:i+batch_size]
    batch_embeddings = processor.model.encode(batch_texts, show_progress_bar=True)
    all_embeddings.append(batch_embeddings)
    print(f"Processed batch {i//batch_size + 1}/{(len(all_texts) + batch_size - 1)//batch_size}")

# Combine all embeddings
embeddings_matrix = np.vstack(all_embeddings)
print(f"Created embeddings matrix with shape: {embeddings_matrix.shape}")

# Normalize embeddings for better similarity search
faiss.normalize_L2(embeddings_matrix)

Creating embeddings...


Batches: 100%|██████████| 1/1 [00:09<00:00,  9.78s/it]


Processed batch 1/53


Batches: 100%|██████████| 1/1 [00:07<00:00,  7.91s/it]


Processed batch 2/53


...

Processed batch 51/53


Batches: 100%|██████████| 1/1 [00:12<00:00, 12.56s/it]


Processed batch 52/53


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.13s/it]

Processed batch 53/53
Created embeddings matrix with shape: (1671, 768)


## Step 5: Filling the database

Generate dense vector embeddings are then being transffered to the FAISS database

In [13]:
# Create FAISS index
print("Creating FAISS index...")

# Use IndexFlatIP for inner product similarity (good for normalized vectors)
index = faiss.IndexFlatIP(processor.embedding_dimension)

# Add embeddings to the index
index.add(embeddings_matrix.astype('float32'))

print(f"FAISS index created with {index.ntotal} vectors")

# Create a mapping from index positions to metadata
index_to_metadata = {i: metadata for i, metadata in enumerate(all_metadata)}

print("Index creation complete!")

Creating FAISS index...
FAISS index created with 1671 vectors
Index creation complete!


## Step 6: FAISS Vector Index Creation

Create a FAISS index using IndexFlatIP for inner product similarity search on normalized vectors. This enables fast semantic similarity queries.

In [14]:
# Save the index and metadata for future use
print("Saving index and metadata...")

# Save FAISS index
faiss.write_index(index, "financial_data.index")

# Save metadata and other components
with open("financial_data_metadata.pkl", "wb") as f:
    pickle.dump({
        'metadata': index_to_metadata,
        'texts': all_texts,
        'holdings_count': len(holdings_texts),
        'trades_count': len(trades_texts),
        'embedding_dimension': processor.embedding_dimension,
        'model_name': 'all-mpnet-base-v2'
    }, f)

print("Saved FAISS index and metadata to disk")

Saving index and metadata...
Saved FAISS index and metadata to disk


## Step 7: Index Persistence

Save the FAISS index and metadata to disk for future use and faster loading.

In [15]:
class FinancialDataSearch:
    def __init__(self, index_path="financial_data.index", metadata_path="financial_data_metadata.pkl"):
        """Load the FAISS index and metadata for searching"""
        self.index = faiss.read_index(index_path)
        
        with open(metadata_path, "rb") as f:
            data = pickle.load(f)
            self.metadata = data['metadata']
            self.texts = data['texts']
            self.holdings_count = data['holdings_count']
            self.trades_count = data['trades_count']
            self.model_name = data['model_name']
        
        self.model = SentenceTransformer(self.model_name)
        
    def search(self, query: str, top_k: int = 5, dataset_filter: str = None) -> List[Dict]:
        """
        Search for similar records
        
        Args:
            query: Natural language query
            top_k: Number of results to return
            dataset_filter: 'holdings', 'trades', or None for both
        """
        # Create embedding for the query
        query_embedding = self.model.encode([query])
        faiss.normalize_L2(query_embedding)
        
        # Search in FAISS index
        scores, indices = self.index.search(query_embedding.astype('float32'), top_k * 2)  # Get more results for filtering
        
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:  # No more results
                break
                
            metadata = self.metadata[idx]
            
            # Apply dataset filter if specified
            if dataset_filter and metadata['dataset_type'] != dataset_filter:
                continue
                
            result = {
                'score': float(score),
                'dataset_type': metadata['dataset_type'],
                'row_index': metadata['row_index'],
                'text': self.texts[idx],
                'metadata': metadata,
                'original_data': metadata['original_data']
            }
            results.append(result)
            
            if len(results) >= top_k:
                break
        
        return results
    
    def get_related_records(self, record_index: int, top_k: int = 5) -> List[Dict]:
        """Find records similar to a specific record by its index"""
        if record_index >= len(self.texts):
            raise ValueError(f"Record index {record_index} out of range")
            
        # Use the embedding of the specified record as query
        query_vector = self.index.reconstruct(record_index).reshape(1, -1)
        scores, indices = self.index.search(query_vector, top_k + 1)  # +1 to exclude self
        
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == record_index:  # Skip the original record
                continue
            if idx == -1:
                break
                
            metadata = self.metadata[idx]
            result = {
                'score': float(score),
                'dataset_type': metadata['dataset_type'],
                'row_index': metadata['row_index'],
                'text': self.texts[idx],
                'metadata': metadata,
                'original_data': metadata['original_data']
            }
            results.append(result)
            
            if len(results) >= top_k:
                break
        
        return results

# Initialize the search interface
searcher = FinancialDataSearch()
print(f"Search interface ready! Index contains {searcher.index.ntotal} records")
print(f"Holdings: {searcher.holdings_count}, Trades: {searcher.trades_count}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 677.73it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Search interface ready! Index contains 1671 records
Holdings: 1022, Trades: 649


## Step 8: Search Interface Implementation

Implement a search interface class that loads the FAISS index and provides semantic search capabilities with metadata filtering.

In [16]:
# Example searches to demonstrate the system
print("=== SEARCH EXAMPLES ===\\n")

# 1. Search for IBM-related transactions
print("1. Searching for IBM-related transactions:")
results = searcher.search("IBM equity transactions", top_k=3)
for i, result in enumerate(results):
    print(f"   Result {i+1}: {result['dataset_type']} - Score: {result['score']:.3f}")
    print(f"   Security: {result['original_data'].get('Name', 'N/A')}")
    print(f"   Portfolio: {result['original_data'].get('PortfolioName', 'N/A')}")
    print(f"   Amount: {result['original_data'].get('TotalCash', 'N/A')}")
    print()

# 2. Search for bond investments  
print("2. Searching for bond investments:")
results = searcher.search("bond investments fixed income", top_k=2, dataset_filter="trades")
for i, result in enumerate(results):
    print(f"   Result {i+1}: Score: {result['score']:.3f}")
    print(f"   Security Type: {result['original_data'].get('SecurityType', 'N/A')}")
    print(f"   Name: {result['original_data'].get('Name', 'N/A')}")
    print()

# 3. Portfolio-specific search
print("3. Portfolio analysis example:")
results = searcher.search("HoldCo 1 portfolio", top_k=2)
for i, result in enumerate(results):
    print(f"   Result {i+1}: {result['dataset_type']} - Score: {result['score']:.3f}")
    print(f"   Security: {result['original_data'].get('Name', 'N/A')}")
    print(f"   Portfolio: {result['original_data'].get('PortfolioName', 'N/A')}")
    print()

=== EXAMPLE SEARCHES ===

1. Searching for IBM-related transactions:
   Result 1: trades - Score: 0.346
   Security: IBM
   Portfolio: Account B
   Amount: 932017148.8

   Result 2: trades - Score: 0.342
   Security: IBM
   Portfolio: Account A
   Amount: 932017148.8

   Result 3: trades - Score: 0.338
   Security: IBM-US
   Portfolio: Optimum Holdings Partners
   Amount: 718513.22

2. Searching for bond investments:
3. Searching for large value transactions:
   Result 1: trades - Score: 0.307
   Total Cash: 2175000.0
   Security: 55279YAA1

   Result 2: trades - Score: 0.307
   Total Cash: 2175000.0
   Security: 55279YAA1

   Result 3: trades - Score: 0.304
   Total Cash: 4500000.0
   Security: BL1156233



## Step 9: Search System Testing

Test the semantic search capabilities with example queries to validate the system functionality.

In [ ]:
import os
from google import genai

class FinancialDataChatBotWithLLM:
    def __init__(self, searcher, holdings_df, trades_df, gemini_api_key=None):
        """
        Initialize the chatbot with FAISS searcher, data, and Gemini API
        """
        self.searcher = searcher
        self.holdings_df = holdings_df
        self.trades_df = trades_df
        
        # Initialize Gemini API using new client pattern
        try:
            api_key = gemini_api_key or os.getenv("GEMINI_API_KEY")
            if api_key:
                self.client = genai.Client(api_key=api_key)
                self.llm_available = True
                print("Gemini API client initialized successfully")
            else:
                self.llm_available = False
                print("Warning: Gemini API key not provided. Using fallback mode.")
        except Exception as e:
            self.llm_available = False
            print(f"Warning: Failed to initialize Gemini API: {e}. Using fallback mode.")
        
        # Prepare fund performance data for context
        self._prepare_fund_data()
        
    def _prepare_fund_data(self):
        """Prepare aggregated fund data for context"""
        self.fund_stats = {}
        
        # Analyze trades data for P&L calculation
        if 'PortfolioName' in self.trades_df.columns:
            for portfolio in self.trades_df['PortfolioName'].unique():
                if pd.notna(portfolio):
                    portfolio_trades = self.trades_df[self.trades_df['PortfolioName'] == portfolio]
                    
                    # Calculate basic statistics
                    total_trades = len(portfolio_trades)
                    total_cash = portfolio_trades['TotalCash'].sum() if 'TotalCash' in portfolio_trades.columns else 0
                    avg_trade_size = portfolio_trades['TotalCash'].mean() if 'TotalCash' in portfolio_trades.columns else 0
                    
                    # Calculate P&L (simplified - buy/sell difference)
                    buys = portfolio_trades[portfolio_trades['TradeTypeName'] == 'Buy']['TotalCash'].sum() if 'TradeTypeName' in portfolio_trades.columns else 0
                    sells = portfolio_trades[portfolio_trades['TradeTypeName'] == 'Sell']['TotalCash'].sum() if 'TradeTypeName' in portfolio_trades.columns else 0
                    net_pnl = sells - buys  # Simplified P&L calculation
                    
                    self.fund_stats[portfolio] = {
                        'total_trades': total_trades,
                        'total_cash_flow': total_cash,
                        'average_trade_size': avg_trade_size,
                        'net_pnl': net_pnl,
                        'buy_volume': buys,
                        'sell_volume': sells
                    }
        
        # Analyze holdings data
        if 'PortfolioName' in self.holdings_df.columns:
            for portfolio in self.holdings_df['PortfolioName'].unique():
                if pd.notna(portfolio):
                    if portfolio not in self.fund_stats:
                        self.fund_stats[portfolio] = {}
                    
                    portfolio_holdings = self.holdings_df[self.holdings_df['PortfolioName'] == portfolio]
                    self.fund_stats[portfolio]['total_holdings'] = len(portfolio_holdings)
    
    def _retrieve_context(self, query, top_k=5):
        """
        Retrieve relevant context from FAISS RAG database
        """
        # Search for relevant documents
        results = self.searcher.search(query, top_k=top_k)
        
        if not results:
            return None, []
        
        # Format context for LLM
        context_parts = []
        relevant_data = []
        
        for i, result in enumerate(results):
            data = result['original_data']
            dataset_type = result['dataset_type']
            score = result['score']
            
            # Create structured context
            if dataset_type == 'trades':
                context_part = f"TRADE RECORD {i+1}:\n"
                context_part += f"- Trade Type: {data.get('TradeTypeName', 'N/A')}\n"
                context_part += f"- Security: {data.get('Name', 'N/A')}\n" 
                context_part += f"- Security Type: {data.get('SecurityType', 'N/A')}\n"
                context_part += f"- Portfolio: {data.get('PortfolioName', 'N/A')}\n"
                context_part += f"- Quantity: {data.get('Quantity', 'N/A')}\n"
                context_part += f"- Price: {data.get('Price', 'N/A')}\n"
                context_part += f"- Total Cash: {data.get('TotalCash', 'N/A')}\n"
                context_part += f"- Trade Date: {data.get('TradeDate', 'N/A')}\n"
                context_part += f"- Custodian: {data.get('CustodianName', 'N/A')}\n"
            else:  # holdings
                context_part = f"HOLDING RECORD {i+1}:\n"
                context_part += f"- Security: {data.get('Name', 'N/A')}\n"
                context_part += f"- Portfolio: {data.get('PortfolioName', 'N/A')}\n"
                # Add other available holding fields
                for key, value in data.items():
                    if key not in ['Name', 'PortfolioName'] and pd.notna(value):
                        context_part += f"- {key}: {value}\n"
            
            context_parts.append(context_part)
            relevant_data.append(data)
        
        context = "\n".join(context_parts)
        return context, relevant_data
    
    def _create_llm_prompt(self, user_query, context, fund_stats):
        """
        Create a comprehensive prompt for the LLM
        """
        prompt = f"""You are a financial data analyst assistant. Answer the user's question based ONLY on the provided financial data context. 

IMPORTANT RULES:
1. Use ONLY the information provided in the context below
2. If the answer cannot be found in the provided data, respond with "Sorry, cannot find the answer in the provided files."
3. Be accurate with numbers and calculations
4. Provide specific details when available
5. Do not make up or assume information not in the context

USER QUESTION: {user_query}

FINANCIAL DATA CONTEXT:
{context}

FUND STATISTICS SUMMARY:
{self._format_fund_stats(fund_stats)}

Please provide a helpful and accurate answer based on this data:"""

        return prompt
    
    def _format_fund_stats(self, fund_stats):
        """Format fund statistics for the prompt"""
        if not fund_stats:
            return "No fund statistics available."
        
        stats_text = ""
        for fund, stats in fund_stats.items():
            stats_text += f"\n{fund}:\n"
            stats_text += f"  - Total Trades: {stats.get('total_trades', 0)}\n"
            stats_text += f"  - Total Holdings: {stats.get('total_holdings', 0)}\n"
            stats_text += f"  - Net P&L: ${stats.get('net_pnl', 0):,.2f}\n"
            stats_text += f"  - Buy Volume: ${stats.get('buy_volume', 0):,.2f}\n"
            stats_text += f"  - Sell Volume: ${stats.get('sell_volume', 0):,.2f}\n"
        
        return stats_text
    
    def ask(self, question):
        """
        Main method: Query → RAG Retrieval → LLM Response
        """
        if not question or not question.strip():
            return "Please ask me a question about the financial data."
        
        try:
            # Step 1: Retrieve relevant context using FAISS embeddings
            context, relevant_data = self._retrieve_context(question, top_k=7)
            
            if not context:
                return "Sorry, cannot find the answer in the provided files."
            
            # Step 2: Use LLM if available, otherwise use fallback
            if self.llm_available:
                try:
                    # Create comprehensive prompt
                    prompt = self._create_llm_prompt(question, context, self.fund_stats)
                    
                    # Generate response using new Gemini client
                    response = self.client.models.generate_content(
                        model="gemini-3-flash-preview",  # or "gemini-3-flash-preview" if available
                        contents=prompt
                    )
                    
                    if response and response.text:
                        return response.text.strip()
                    else:
                        return "Sorry, cannot find the answer in the provided files."
                        
                except Exception as e:
                    print(f"LLM Error: {e}")
                    return self._fallback_response(question, relevant_data)
            else:
                # Fallback mode without LLM
                return self._fallback_response(question, relevant_data)
        
        except Exception as e:
            print(f"Error: {e}")
            return "Sorry, cannot find the answer in the provided files."
    
    def _fallback_response(self, question, relevant_data):
        """Fallback response when LLM is not available"""
        if not relevant_data:
            return "Sorry, cannot find the answer in the provided files."
        
        response = f"Found {len(relevant_data)} relevant records:\n\n"
        
        for i, data in enumerate(relevant_data):
            response += f"{i+1}. "
            if 'TradeTypeName' in data:
                response += f"{data.get('TradeTypeName', 'N/A')} - {data.get('Name', 'N/A')} "
                response += f"(Portfolio: {data.get('PortfolioName', 'N/A')}, "
                response += f"Amount: ${data.get('TotalCash', 0):,.2f})\n"
            else:
                response += f"Holding - {data.get('Name', 'N/A')} "
                response += f"(Portfolio: {data.get('PortfolioName', 'N/A')})\n"
        
        return response

# Initialize the enhanced chatbot with new Gemini API
try:
    # The client gets the API key from the environment variable or passed parameter
    api_key = os.getenv('GEMINI_API_KEY') 
    enhanced_chatbot = FinancialDataChatBotWithLLM(searcher, holdings_df, trades_df, api_key)
    print("Enhanced Financial Data ChatBot with Gemini LLM is ready")
    print("Features: RAG + FAISS Embeddings + Google Gemini Flash")
except Exception as e:
    enhanced_chatbot = FinancialDataChatBotWithLLM(searcher, holdings_df, trades_df)
    print("Financial Data ChatBot is ready (fallback mode)")
    print(f"Note: {e}")

print("\\nExample questions:")
print("- 'How many IBM trades were executed and what was the total value?'")
print("- 'Which portfolio has the best performance in terms of P&L?'")
print("- 'Show me all bond transactions for HoldCo 1'")
print("- 'What is the average trade size for ClientA?'")

Gemini API client initialized successfully
Enhanced Financial Data ChatBot with Gemini LLM is ready
Features: RAG + FAISS Embeddings + Google Gemini Flash
\nExample questions:
- 'How many IBM trades were executed and what was the total value?'
- 'Which portfolio has the best performance in terms of P&L?'
- 'Show me all bond transactions for HoldCo 1'
- 'What is the average trade size for ClientA?'


## Step 10: RAG-Powered Chatbot Implementation

Implement an intelligent chatbot that combines FAISS semantic search with Google Gemini LLM for comprehensive financial data analysis.

In [33]:
# Interactive Chat Interface
def start_enhanced_chat():
    """Start an interactive chat session with the enhanced LLM-powered bot"""
    print("=" * 70)
    print("ENHANCED FINANCIAL DATA CHATBOT with GEMINI LLM")
    print("=" * 70)
    print("Powered by: RAG + FAISS Embeddings + Google Gemini Pro")
    print("Type 'exit', 'quit', or 'bye' to end the conversation.")
    print("=" * 70)
    
    while True:
        try:
            user_input = input("\\nYou: ").strip()
            
            if user_input.lower() in ['exit', 'quit', 'bye', '']:
                print("Enhanced ChatBot: Goodbye! Thanks for using the Enhanced Financial Data ChatBot!")
                break
            
            print("Searching financial data...")
            response = enhanced_chatbot.ask(user_input)
            print(f"\\nEnhanced ChatBot: {response}")
            
        except KeyboardInterrupt:
            print("\\n\\nEnhanced ChatBot: Goodbye! Thanks for using the Enhanced Financial Data ChatBot!")
            break
        except Exception as e:
            print(f"\\nEnhanced ChatBot: Sorry, I encountered an error. Please try again.")

## Step 11: Chatbot Testing and Validation

Test the enhanced chatbot with complex queries to demonstrate RAG capabilities.

In [34]:
# Test the enhanced chatbot with complex questions
print("TESTING ENHANCED CHATBOT WITH COMPLEX QUESTIONS")
print("=" * 60)

complex_test_questions = [
    "How many IBM trades were executed and what was the total value?",
    "Which portfolio has the best performance in terms of P&L?",
    "Show me all bond transactions and calculate their average value",
    "Compare the trading activity between HoldCo 1 and ClientA",
    "What are the largest equity transactions in the database?"
]

for question in complex_test_questions[:3]:  # Test first 3 questions
    print(f"\\nComplex Question: {question}")
    print("Processing with RAG + LLM...")
    answer = enhanced_chatbot.ask(question)
    print(f"Enhanced Answer: {answer}")
    print("-" * 60)

TESTING ENHANCED CHATBOT WITH COMPLEX QUESTIONS
\nComplex Question: How many IBM trades were executed and what was the total value?
Processing with RAG + LLM...
Enhanced Answer: Based on the financial data provided, there were a total of **7 IBM trades** executed, with a total value (Total Cash) of **$229,353,822.90**.

The breakdown of these trades is as follows:

*   **Trade 1 (Equity):** $718,513.22 (Buy 5,000 IBM-US)
*   **Trade 2 (Equity):** $4,971,643.48 (Buy 51,787 IBM-US)
*   **Trade 3 (Option):** $184,000.00 (Buy 200 IBM 220117P00150000)
*   **Trade 4 (Equity):** $52,840,024.80 (Buy 500,000 IBM-US)
*   **Trade 5 (Equity):** $52,840,024.80 (Sell 500,000 IBM-US)
*   **Trade 6 (Equity):** $67,264,687.40 (Buy 500,000 IBM-US)
*   **Trade 7 (Equity):** $50,534,929.20 (Buy 500,000 IBM-US)

**Total Calculation:**
$718,513.22 + $4,971,643.48 + $184,000.00 + $52,840,024.80 + $52,840,024.80 + $67,264,687.40 + $50,534,929.20 = **$229,353,822.90**
------------------------------------------

## Conclusion

This notebook demonstrates a complete RAG (Retrieval-Augmented Generation) system for financial data analysis:

### Key Features Implemented:
1. **Document Preprocessing**: Structured text conversion of tabular financial data
2. **Semantic Embeddings**: SentenceTransformer embeddings for semantic similarity
3. **FAISS Vector Database**: Efficient similarity search with normalized vectors
4. **RAG Pipeline**: Context retrieval combined with LLM generation
5. **Financial Intelligence**: Specialized chatbot for investment analysis

### System Performance:
- Handles both holdings and trades data seamlessly
- Provides semantic search across financial records
- Supports complex queries with intelligent LLM responses
- Maintains data accuracy and traceability

### Usage:
1. Set up your Gemini API key
2. Run the notebook cells sequentially
3. Use `enhanced_chatbot.ask("your question")` for queries
4. Start interactive chat with `start_enhanced_chat()`